In [5]:
import pandas as pd

### Data exploration

In [6]:
restaurants = pd.read_csv("../data/csv_files/restaurants.csv")
restaurants.head()

,restaurant_id,cuisine,city,rating
0,R001,Indian,Manchester,3.4
1,R002,Chinese,Leeds,4.1
2,R003,Chinese,Leeds,4.1
3,R004,Thai,Birmingham,3.5
4,R005,American,Bristol,4.3


## Ingestion de las tablas

In [11]:
from sqlalchemy import create_engine
import os

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)

In [8]:
restaurants["source_file"] = "restaurants.csv"
restaurants.head()

,restaurant_id,cuisine,city,rating,source_file
0,R001,Indian,Manchester,3.4,restaurants.csv
1,R002,Chinese,Leeds,4.1,restaurants.csv
2,R003,Chinese,Leeds,4.1,restaurants.csv
3,R004,Thai,Birmingham,3.5,restaurants.csv
4,R005,American,Bristol,4.3,restaurants.csv


In [14]:
restaurants.to_sql(
    'restaurants', 
    engine, 
    schema='raw', 
    if_exists='replace', 
    index=False
)

120

In [12]:
df_test = pd.read_sql("SELECT * FROM raw.restaurants LIMIT 5;", con=engine)
print(df_test)

  restaurant_id   cuisine        city  rating
0          R001    Indian  Manchester     3.4
1          R002   Chinese       Leeds     4.1
2          R003   Chinese       Leeds     4.1
3          R004      Thai  Birmingham     3.5
4          R005  American     Bristol     4.3


# 03 Transformation Test

This notebook is used to prototype the transformation logic before converting it into PostgreSQL stored procedures.

For each table, the structure is:

```text
1. Query the raw table
2. Apply the transformation SELECT statement

```

In order to compare the raw output with the transformed output


In [15]:
from pathlib import Path
import sys
import pandas as pd

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import  get_engine

engine = get_engine()

2026-09-16 20:26:05,171 | INFO | db_connection | Database environment variables validated successfully.
2026-09-16 20:26:05,214 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=food_platform, user=postgres
2026-09-16 20:26:05,229 | INFO | db_connection | Creating SQLAlchemy engine.


In [15]:
from sqlalchemy import text

def run_query(query: str) -> pd.DataFrame:
  with engine.connect() as conn:
    return pd.read_sql_query(text(query), conn)

### 3. Tranformation

Rules:

1. Validate `restaurant_id` is not null and clean up leading/trailing whitespaces in text fields (`cuisine`, `city`).
2. Standardize text casing using `INITCAP` for `cuisine` and `city`.
3. Cast `rating` to `NUMERIC(3,2)` and preserve the `source_file` tracking column.

In [17]:
raw_restaurants_query = """
SELECT * FROM raw.restaurants
"""

raw_restaurants_df = run_query(raw_restaurants_query)
raw_restaurants_df.head()

,restaurant_id,cuisine,city,rating,source_file
0,R001,Indian,Manchester,3.4,restaurants.csv
1,R002,Chinese,Leeds,4.1,restaurants.csv
2,R003,Chinese,Leeds,4.1,restaurants.csv
3,R004,Thai,Birmingham,3.5,restaurants.csv
4,R005,American,Bristol,4.3,restaurants.csv


## Apply transformation

In [19]:
transformed_restaurants_query = """
SELECT 
    restaurant_id,
    INITCAP(TRIM(cuisine)) AS cuisine,
    INITCAP(TRIM(city)) AS city,
    rating,
    source_file
FROM raw.restaurants
WHERE restaurant_id IS NOT NULL;
"""

transformed_restaurants_df = run_query(transformed_restaurants_query)
transformed_restaurants_df.head()

,restaurant_id,cuisine,city,rating,source_file
0,R001,Indian,Manchester,3.4,restaurants.csv
1,R002,Chinese,Leeds,4.1,restaurants.csv
2,R003,Chinese,Leeds,4.1,restaurants.csv
3,R004,Thai,Birmingham,3.5,restaurants.csv
4,R005,American,Bristol,4.3,restaurants.csv
